In [ ]:
import sys
sys.path.insert(0, '../..')


Best network across time in task based fMRI in RAW

In [ ]:
import pandas as pd
from scipy.special import softmax
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

df = None
pop = "test"
data, prediction, labels, id1, id2, rate, scores = visualization._get_predictions(pop, "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
scores = np.array([score.numpy() for score in scores])
for label in range(3):
    mask = np.argmax(softmax(prediction.numpy().sum(1), axis=-1), axis=-1) == label
    mask2 = mask.copy()

    mask = np.ones_like(mask) == 1

    x_norm = data.numpy()[mask][scores[mask, 0] < 35/80] / np.sqrt(np.sum(data.numpy()[mask][scores[mask, 0] < 35/80] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T)
    x_network = x_network.reshape(-1, 25, 18)[mask2[scores[mask, 0] < 35/80]]
    tmp = np.abs(x_network).mean(0)
    sign = (x_network).mean(0)
    tmp[..., 5] = 0
    networkt1 = (network.iloc[np.argmax(tmp, axis=-1), 0])
    networkt1s = (sign.mean(0)[np.argmax(tmp, axis=-1)]) > 0
    
    x_norm = data.numpy()[mask][scores[mask, 0] > 52/80] / np.sqrt(np.sum(data.numpy()[mask][scores[mask, 0] > 52/80] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(-1, 25, 18)
    x_network = x_network.reshape(-1, 25, 18)[mask2[scores[mask, 0] > 52/80]]
    tmp = np.abs(x_network).mean(0)
    sign = (x_network).mean(0)
    tmp[..., 5] = 0
    networkt3 = (network.iloc[np.argmax(tmp, axis=-1), 0])
    networkt3s = (sign.mean(0)[np.argmax(tmp, axis=-1)]) > 0

    df = pd.DataFrame({'Network-T1':networkt1.values, 'Sign-T1':networkt1s, 'Network-T3':networkt3.values, 'Sign-T3':networkt3s})
    print(df)

In [ ]:
import pandas as pd
from scipy.special import softmax
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

df = None
pop = "test"
data, prediction, labels, id1, id2, rate, scores = visualization._get_predictions(pop, "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
scores = np.array([score.numpy() for score in scores])
for label in range(3):
    mask = np.argmax(softmax(prediction.numpy().sum(1), axis=-1), axis=-1) == label
    mask2 = mask.copy()

    mask = np.ones_like(mask) == 1

    x_norm = data.numpy()[mask][scores[mask, 0] < 35/80] / np.sqrt(np.sum(data.numpy()[mask][scores[mask, 0] < 35/80] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T)
    x_network = x_network.reshape(-1, 25, 18)[mask2[scores[mask, 0] < 35/80]]

    transitions = np.argmax(x_network, axis=2)
    flux_matrix = np.zeros((18, 18), dtype=int)
    for subject_transitions in transitions:
        from_idx = subject_transitions[:-1]
        to_idx = subject_transitions[1:]
        np.add.at(flux_matrix, (from_idx, to_idx), 1)

    flux_matrix = flux_matrix.astype(float)

    # Knowing node proba
    row_sums = flux_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    flux_matrix = flux_matrix / row_sums

    # Full proba
    #flux_matrix = flux_matrix / flux_matrix.sum()

    #flux_matrix = np.where(flux_matrix < np.quantile(flux_matrix, .80), 0, flux_matrix)
    no_diag_flux = flux_matrix.copy()
    np.fill_diagonal(no_diag_flux, 0)
    nodes_mask = (no_diag_flux > 0).any(axis=1)
    flux_matrix = flux_matrix[nodes_mask, :]
    flux_matrix = flux_matrix[:, nodes_mask]
    labels_networks = network.iloc[nodes_mask, 0].to_numpy()
    plt.imshow(flux_matrix)
    plt.show()
    
    import networkx as nx
    graph = nx.from_numpy_array(flux_matrix, create_using=nx.DiGraph)
    
    plt.figure(figsize=(12, 9))
    labels_networks = {i: labels_networks[i] for i in graph.nodes()}
    edges = graph.edges(data=True)
    edge_weights = [d['weight'] * 5 for (_, _, d) in edges]
    pos = nx.spring_layout(graph, weight='weight', seed=42)

    node_degrees = dict(graph.degree())
    node_sizes = [300 + 50 * node_degrees[n] for n in graph.nodes()]

    edge_labels = {(i, j): f"{d['weight']:.2f}" for i, j, d in edges}

    nx.draw(graph, pos, with_labels=True, edge_color='gray', labels=labels_networks, node_color='lightblue', node_size=node_sizes, font_size=10)

    nx.draw_networkx_edges(graph, pos, edgelist=graph.edges(), edge_color='gray',
                        width=edge_weights, arrows=True, arrowstyle='-|>', arrowsize=15)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=9, label_pos=0.5)

    plt.title("Graph Visualization with Edge Weights")
    plt.show()

    pagerank = nx.pagerank(graph, weight='weight')
    top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:3]

    print("\n🔝 Top 3 nodes by PageRank:")
    for node, value in top_pagerank:
        label = labels_networks[graph.nodes[node].get("label", node)]
        print(f"  {label}: PageRank = {value:.4f}")
    
    x_norm = data.numpy()[mask][scores[mask, 0] > 52/80] / np.sqrt(np.sum(data.numpy()[mask][scores[mask, 0] > 52/80] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(-1, 25, 18)
    x_network = x_network.reshape(-1, 25, 18)[mask2[scores[mask, 0] > 52/80]]

    transitions = np.argmax(x_network, axis=2)
    flux_matrix = np.zeros((18, 18), dtype=int)
    for subject_transitions in transitions:
        from_idx = subject_transitions[:-1]
        to_idx = subject_transitions[1:]
        np.add.at(flux_matrix, (from_idx, to_idx), 1)

    flux_matrix = flux_matrix / x_network.shape[0]
    no_diag_flux = flux_matrix.copy()
    np.fill_diagonal(no_diag_flux, 0)
    nodes_mask = (no_diag_flux > 0).any(axis=1)
    flux_matrix = flux_matrix[nodes_mask, :]
    flux_matrix = flux_matrix[:, nodes_mask]
    labels = network.iloc[nodes_mask, 0].to_numpy()
    plt.imshow(flux_matrix)
    plt.show()
    
    import networkx as nx
    graph = nx.from_numpy_array(flux_matrix)
    edges = graph.edges(data=True)
    edge_weights = np.array([data['weight'] for _, _, data in edges])
    pos = nx.spring_layout(graph, weight='weight', seed=42)
    
    plt.figure(figsize=(12, 9))
    labels = {i: labels[i] for i in graph.nodes()}
    nx.draw(graph, pos, with_labels=True, edge_color='gray', labels=labels, node_color='lightblue', node_size=100, font_size=10)

    edge_labels = nx.get_edge_attributes(graph, 'weight')
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels)

    plt.title("Graph Visualization with Edge Weights")
    plt.show()

Rest Val FINA - PSWQ

In [ ]:
import pandas as pd
import numpy as np
from scipy.special import softmax

_, _, _, _, id2v, _, _ = visualization._get_predictions("val", "full")

# Load the network data
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

# Assume visualization._get_predictions is a function that returns the required data
data, prediction, id1, id2, scores = visualization._get_predictions("pred", "full")
id1, id2, id2v = np.array(id1), np.array(id2).astype(np.int64), np.array(id2v).astype(np.int64)
keep = np.isin(id2, id2v)
data, prediction, id1, id2, scores = data.numpy()[keep], prediction.numpy()[keep], id1[keep], id2[keep], scores
scores = np.array([score.numpy() for score in scores])[keep]

# Normalize the data
x_norm = data / np.sqrt(np.sum(data ** 2, axis=1, keepdims=True))
x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(24, 480, 18)

# Initialize a DataFrame to store the results
results = pd.DataFrame(columns=['Label', 'Network-T1-Init', 'Network-T1-Maintain', 'Network-T3-Init', 'Network-T3-Maintain'])

# Define the cut-offs for T1 and T3
t1_cutoff = 35
t3_cutoff = 54

# Filter the population based on the cut-offs
t1_indices = np.where(scores[:, 0] < t1_cutoff/80)[0]
t3_indices = np.where(scores[:, 0] > t3_cutoff/80)[0]

# Process each label
for label in range(3):
    mask = np.argmax(softmax(prediction, axis=-1), axis=-1) == label

    # Process for T1
    flux_matrix = np.zeros((18, 18), dtype=int)
    for i in range(len(t1_indices)):
        maskg = np.where(np.diff(mask[t1_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t1_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vector = x_network[t1_indices[i]][maskg[j]:maskg[j + 1]]
                transitions = np.argmax(np.abs(vector), axis=-1)
                from_idx = transitions[:-1]
                to_idx = transitions[1:]
                np.add.at(flux_matrix, (from_idx, to_idx), 1)
    flux_matrix = flux_matrix.astype(float)
    flux_matrix = flux_matrix / flux_matrix.sum()
    no_diag_flux = flux_matrix.copy()
    np.fill_diagonal(no_diag_flux, 0)
    nodes_mask = (no_diag_flux > 0).any(axis=1)
    flux_matrix = flux_matrix[nodes_mask, :]
    flux_matrix = flux_matrix[:, nodes_mask]
    labels_networks = network.iloc[nodes_mask, 0].to_numpy()
    plt.imshow(flux_matrix)
    plt.show()
    
    import networkx as nx
    graph = nx.from_numpy_array(flux_matrix, create_using=nx.DiGraph)
    
    plt.figure(figsize=(12, 9))
    labels_networks = {i: labels_networks[i] for i in graph.nodes()}
    edges = graph.edges(data=True)
    edge_weights = [d['weight'] * 5 for (_, _, d) in edges]
    pos = nx.spring_layout(graph, weight='weight', seed=42)

    edge_labels = {(i, j): f"{d['weight']:.2f}" for i, j, d in edges}

    nx.draw(graph, pos, with_labels=True, edge_color='gray', labels=labels_networks, node_color='lightblue', node_size=300, font_size=10)

    nx.draw_networkx_edges(graph, pos, edgelist=graph.edges(), edge_color='gray',
                        width=edge_weights, arrows=True, arrowstyle='-|>', arrowsize=15)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=9, label_pos=0.5)

    plt.title("Graph Visualization with Edge Weights")
    plt.show()

    pagerank = nx.pagerank(graph, weight='weight')
    top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]

    print("\n🔝 Top 3 nodes by PageRank:")
    for node, value in top_pagerank:
        label = labels_networks[graph.nodes[node].get("label", node)]
        print(f"  {label}: PageRank = {value:.4f}")


    # Process for T3
    flux_matrix = np.zeros((18, 18), dtype=int)
    for i in range(len(t3_indices)):
        maskg = np.where(np.diff(mask[t3_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t3_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vector = x_network[t3_indices[i]][maskg[j]:maskg[j + 1]]
                transitions = np.argmax(np.abs(vector), axis=-1)
                from_idx = transitions[:-1]
                to_idx = transitions[1:]
                np.add.at(flux_matrix, (from_idx, to_idx), 1)
    flux_matrix = flux_matrix.astype(float)
    flux_matrix = flux_matrix / flux_matrix.sum()
    no_diag_flux = flux_matrix.copy()
    np.fill_diagonal(no_diag_flux, 0)
    nodes_mask = (no_diag_flux > 0).any(axis=1)
    flux_matrix = flux_matrix[nodes_mask, :]
    flux_matrix = flux_matrix[:, nodes_mask]
    labels_networks = network.iloc[nodes_mask, 0].to_numpy()
    plt.imshow(flux_matrix)
    plt.show()
    
    import networkx as nx
    graph = nx.from_numpy_array(flux_matrix, create_using=nx.DiGraph)
    
    plt.figure(figsize=(12, 9))
    labels_networks = {i: labels_networks[i] for i in graph.nodes()}
    edges = graph.edges(data=True)
    edge_weights = [d['weight'] * 5 for (_, _, d) in edges]
    pos = nx.spring_layout(graph, weight='weight', seed=42)

    edge_labels = {(i, j): f"{d['weight']:.2f}" for i, j, d in edges}

    nx.draw(graph, pos, with_labels=True, edge_color='gray', labels=labels_networks, node_color='lightblue', node_size=300, font_size=10)

    nx.draw_networkx_edges(graph, pos, edgelist=graph.edges(), edge_color='gray',
                        width=edge_weights, arrows=True, arrowstyle='-|>', arrowsize=15)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=9, label_pos=0.5)

    plt.title("Graph Visualization with Edge Weights")
    plt.show()

    pagerank = nx.pagerank(graph, weight='weight')
    top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]

    print("\n🔝 Top 3 nodes by PageRank:")
    for node, value in top_pagerank:
        label = labels_networks[graph.nodes[node].get("label", node)]
        print(f"  {label}: PageRank = {value:.4f}")

print(pos, labels_networks, pagerank, np.diag(flux_matrix))

In [ ]:
import pandas as pd
import numpy as np
from scipy.special import softmax

_, _, _, _, id2v, _, _ = visualization._get_predictions("val", "full")

# Load the network data
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

# Assume visualization._get_predictions is a function that returns the required data
data, prediction, id1, id2, scores = visualization._get_predictions("pred", "full")
id1, id2, id2v = np.array(id1), np.array(id2).astype(np.int64), np.array(id2v).astype(np.int64)
keep = np.isin(id2, id2v)
data, prediction, id1, id2, scores = data.numpy()[keep], prediction.numpy()[keep], id1[keep], id2[keep], scores
scores = np.array([score.numpy() for score in scores])[keep]

# Normalize the data
x_norm = data / np.sqrt(np.sum(data ** 2, axis=1, keepdims=True))
x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(24, 480, 18)

# Initialize a DataFrame to store the results
results = pd.DataFrame(columns=['Label', 'Network-T1-Init', 'Network-T1-Maintain', 'Network-T3-Init', 'Network-T3-Maintain'])

# Define the cut-offs for T1 and T3
t1_cutoff = 35
t3_cutoff = 54

# Filter the population based on the cut-offs
t1_indices = np.where(scores[:, 0] < t1_cutoff/80)[0]
t3_indices = np.where(scores[:, 0] > t3_cutoff/80)[0]

# Process each label
for label in range(3):
    mask = np.argmax(softmax(prediction, axis=-1), axis=-1) == label

    # Process for T1
    vectors = []
    for i in range(len(t1_indices)):
        maskg = np.where(np.diff(mask[t1_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t1_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vectors.append(x_network[t1_indices[i]][maskg[j]:maskg[j + 1]])

    if vectors:
        print(len(vectors), len(t1_indices))
        vectors_init = np.concatenate([vector[:11, :] for vector in vectors], axis=0)
        print(vectors_init.shape)
        vectors_init = np.abs(vectors_init).mean(0)
        vectors_init = np.argsort(vectors_init).reshape(1, 18)
        vectors_maintain = np.concatenate([vector[11:] for vector in vectors if vector.shape[0] > 10], axis=0)
        vectors_maintain = np.abs(vectors_maintain).mean(0)
        vectors_maintain = np.argsort(vectors_maintain).reshape(1, 18)
        final_init_t1 = network.iloc[:, 0].to_numpy()[vectors_init[:, -5:]].flatten()
        final_maintain_t1 = network.iloc[:, 0].to_numpy()[vectors_maintain[:, -5:]].flatten()
    else:
        final_init_t1 = final_maintain_t1 = np.array([])

    # Process for T3
    vectors = []
    for i in range(len(t3_indices)):
        maskg = np.where(np.diff(mask[t3_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t3_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vectors.append(x_network[t3_indices[i]][maskg[j]:maskg[j + 1]])

    if vectors:
        vectors_init = np.concatenate([vector[:11] for vector in vectors], axis=0)
        vectors_init = np.abs(vectors_init).mean(0)
        vectors_init = np.argsort(vectors_init).reshape(1, 18)
        vectors_maintain = np.concatenate([vector[11:] for vector in vectors if vector.shape[0] > 10], axis=0)
        vectors_maintain = np.abs(vectors_maintain).mean(0)
        vectors_maintain = np.argsort(vectors_maintain).reshape(1, 18)
        final_init_t3 = network.iloc[:, 0].to_numpy()[vectors_init[:, -5:]].flatten()
        final_maintain_t3 = network.iloc[:, 0].to_numpy()[vectors_maintain[:, -5:]].flatten()
    else:
        final_init_t3 = final_maintain_t3 = np.array([])

    # Append the results to the DataFrame
    results = pd.concat([results, pd.DataFrame({
        'Label': np.array(label).repeat(5),
        'Network-T1-Init': final_init_t1,
        'Network-T1-Maintain': final_maintain_t1,
        'Network-T3-Init': final_init_t3,
        'Network-T3-Maintain': final_maintain_t3
    })], ignore_index=True)

results


DF build

In [ ]:
fina = pd.read_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "FINA", "Public", "Analysis", "misc", "FINA_2025_01_17.csv"))
cols = ["pswq_total", "rsq_total", "hars_score", "age", "sex", "madrs_total", "education", "fina_mmse_3ms"]
fina["age"] = fina["age_at_mrscan"]
fina["gender"] = fina["sex"]
fina["Vault_UID"] = fina["Vault_ScanID"]

_, _, _, _, id2v, _, _ = visualization._get_predictions("val", "full")
data, prediction, id1, id2, scores = visualization._get_predictions("pred", "full")
id1, id2, id2v = np.array(id1), np.array(id2), np.array(id2v)
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
keep = np.isin(id2, id2v)
data, prediction, id1, id2, scores = data.numpy()[keep], prediction.numpy()[keep], id1[keep], id2[keep], scores
scores = np.array([score.numpy() for score in scores])[keep]

fina = fina[fina.Vault_UID.isin(id2v)]
fina = fina[["Vault_UID"]+cols]
position = fina.Vault_UID.apply(lambda x: id2v.tolist().index(x)).to_numpy().argsort()
fina = fina.iloc[position]
print(len(fina[fina.sex == 1]))
print(fina.Vault_UID.to_numpy(), id2v)
print(fina.pswq_total.to_numpy(), scores[:, 0]*80)
fina.describe()

In [ ]:
raw = pd.read_csv(os.path.join("/mnt/argo/Workspaces", "Staff", "Jacques-Yves_Campion", "Public", "Project-1_WorryInduction", "Data", "RAW_JY_Data_Request_2.10.25.csv"))
data, prediction, labels, id1, id2, rate, scores = visualization._get_predictions("test", "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
scores = np.array([score.numpy() for score in scores])

raw["Vault_UID"] = raw.raw_id.str.replace("RAW_", "").astype(np.int64)
raw = raw[raw.Vault_UID.isin(id2.astype(np.int64))]
cols = ["pswq_total", "rsq_total", "hars_score", "age", "gender", "hamd_total"]
#, "pss_score", "psqi_global", "phq9_score", "age", "gender", "race", "cirstot", "gad7_total", "hamd_total", "neoffi_score_e1", "neoffi_score_o1", "neoffi_score_a1", "neoffi_score_c1"]
raw = raw[["Vault_UID"]+cols]

raw["sex"] = raw.gender

position = raw.Vault_UID.apply(lambda x: id2.astype(np.int64).tolist().index(x)).to_numpy().argsort()
raw = raw.iloc[position]

print(len(raw[raw.sex == 1]))
print(raw.Vault_UID.to_numpy(), id2.astype(np.int64))
print(raw.pswq_total.to_numpy(), (scores[:, 0]*80)[::34])

raw.describe()

In [ ]:
import pandas as pd
from scipy.special import softmax
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

dataf = None
pop = "test"
data, prediction, labels, id1, id2, rate, scores = visualization._get_predictions(pop, "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
scores = np.array([score.numpy() for score in scores])

ages = raw.age.to_numpy()
t1_cutoff = np.quantile(ages, .33)
t3_cutoff = np.quantile(ages, .66)

for label in range(3):
    mask = np.argmax(softmax(prediction.numpy().sum(1), axis=-1), axis=-1) == label
    mask2 = mask.copy()

    mask = np.ones_like(mask) == 1
    nblocks = 34

    x_norm = data.numpy()[mask][ages.repeat(nblocks) < t1_cutoff] / np.sqrt(np.sum(data.numpy()[mask][ages.repeat(nblocks) < t1_cutoff] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T)
    x_network = x_network.reshape(-1, 25, 18)[mask2[ages.repeat(nblocks) < t1_cutoff]]
    tmp = np.abs(x_network).mean(0)
    sign = (x_network).mean(0)
    tmp[..., 5] = 0
    networkt1 = (network.iloc[np.argmax(tmp, axis=-1), 0])
    networkt1s = (sign.mean(0)[np.argmax(tmp, axis=-1)]) > 0
    
    x_norm = data.numpy()[mask][ages.repeat(nblocks) > t3_cutoff] / np.sqrt(np.sum(data.numpy()[mask][ages.repeat(nblocks) > t3_cutoff] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(-1, 25, 18)
    x_network = x_network.reshape(-1, 25, 18)[mask2[ages.repeat(nblocks) > t3_cutoff]]
    tmp = np.abs(x_network).mean(0)
    sign = (x_network).mean(0)
    tmp[..., 5] = 0
    networkt3 = (network.iloc[np.argmax(tmp, axis=-1), 0])
    networkt3s = (sign.mean(0)[np.argmax(tmp, axis=-1)]) > 0

    dataf = pd.DataFrame({'Network-Y':networkt1.values, 'Sign-Y':networkt1s, 'Network-O':networkt3.values, 'Sign-O':networkt3s})
    print(dataf)

Rest Val FINA - Age

In [ ]:
import pandas as pd
import numpy as np
from scipy.special import softmax

_, _, _, _, id2v, _, _ = visualization._get_predictions("val", "full")

# Load the network data
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

# Assume visualization._get_predictions is a function that returns the required data
data, prediction, id1, id2, scores = visualization._get_predictions("pred", "full")
id1, id2, id2v = np.array(id1), np.array(id2).astype(np.int64), np.array(id2v).astype(np.int64)
keep = np.isin(id2, id2v)
data, prediction, id1, id2, scores = data.numpy()[keep], prediction.numpy()[keep], id1[keep], id2[keep], scores
scores = np.array([score.numpy() for score in scores])[keep]

# Normalize the data
x_norm = data / np.sqrt(np.sum(data ** 2, axis=1, keepdims=True))
x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(24, 480, 18)

# Initialize a DataFrame to store the results
results = pd.DataFrame(columns=['Label', 'Network-Y-Init', 'Network-Y-Maintain', 'Network-O-Init', 'Network-O-Maintain'])

# Define the cut-offs for T1 and T3
ages = fina.age

# Filter the population based on the cut-offs
t1_indices = np.where(ages < t1_cutoff)[0]
t3_indices = np.where(ages > t3_cutoff)[0]
print(t1_cutoff, t3_cutoff)
print(len(t1_indices), len(t3_indices))

# Process each label
for label in range(3):
    mask = np.argmax(softmax(prediction, axis=-1), axis=-1) == label

    # Process for T1
    vectors = []
    for i in range(len(t1_indices)):
        maskg = np.where(np.diff(mask[t1_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t1_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vectors.append(x_network[t1_indices[i]][maskg[j]:maskg[j + 1]])

    if vectors:
        vectors_init = np.concatenate([vector[:11, :] for vector in vectors], axis=0)
        vectors_init = np.abs(vectors_init).mean(0)
        vectors_init = np.argsort(vectors_init).reshape(1, 18)
        vectors_maintain = np.concatenate([vector[11:] for vector in vectors if vector.shape[0] > 10], axis=0)
        vectors_maintain = np.abs(vectors_maintain).mean(0)
        vectors_maintain = np.argsort(vectors_maintain).reshape(1, 18)
        final_init_t1 = network.iloc[:, 0].to_numpy()[vectors_init[:, -5:]].flatten()
        final_maintain_t1 = network.iloc[:, 0].to_numpy()[vectors_maintain[:, -5:]].flatten()
    else:
        final_init_t1 = final_maintain_t1 = np.array([])

    # Process for T3
    vectors = []
    for i in range(len(t3_indices)):
        maskg = np.where(np.diff(mask[t3_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t3_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vectors.append(x_network[t3_indices[i]][maskg[j]:maskg[j + 1]])

    if vectors:
        vectors_init = np.concatenate([vector[:11] for vector in vectors], axis=0)
        vectors_init = np.abs(vectors_init).mean(0)
        vectors_init = np.argsort(vectors_init).reshape(1, 18)
        vectors_maintain = np.concatenate([vector[11:] for vector in vectors if vector.shape[0] > 10], axis=0)
        vectors_maintain = np.abs(vectors_maintain).mean(0)
        vectors_maintain = np.argsort(vectors_maintain).reshape(1, 18)
        final_init_t3 = network.iloc[:, 0].to_numpy()[vectors_init[:, -5:]].flatten()
        final_maintain_t3 = network.iloc[:, 0].to_numpy()[vectors_maintain[:, -5:]].flatten()
    else:
        final_init_t3 = final_maintain_t3 = np.array([])

    # Append the results to the DataFrame
    results = pd.concat([results, pd.DataFrame({
        'Label': np.array(label).repeat(5),
        'Network-Y-Init': final_init_t1,
        'Network-Y-Maintain': final_maintain_t1,
        'Network-O-Init': final_init_t3,
        'Network-O-Maintain': final_maintain_t3
    })], ignore_index=True)

results


In [ ]:
import pandas as pd
from scipy.special import softmax
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

dataf = None
pop = "test"
data, prediction, labels, id1, id2, rate, scores = visualization._get_predictions(pop, "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
scores = np.array([score.numpy() for score in scores])

sex = raw.sex.to_numpy()

for label in range(3):
    mask = np.argmax(softmax(prediction.numpy().sum(1), axis=-1), axis=-1) == label
    mask2 = mask.copy()

    mask = np.ones_like(mask) == 1
    nblocks = 34

    x_norm = data.numpy()[mask][sex.repeat(nblocks) == 1] / np.sqrt(np.sum(data.numpy()[mask][sex.repeat(nblocks) == 1] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T)
    x_network = x_network.reshape(-1, 25, 18)[mask2[sex.repeat(nblocks) == 1]]
    tmp = np.abs(x_network).mean(0)
    sign = (x_network).mean(0)
    tmp[..., 5] = 0
    networkt1 = (network.iloc[np.argmax(tmp, axis=-1), 0])
    networkt1s = (sign.mean(0)[np.argmax(tmp, axis=-1)]) > 0
    
    x_norm = data.numpy()[mask][sex.repeat(nblocks) == 2] / np.sqrt(np.sum(data.numpy()[mask][sex.repeat(nblocks) == 2] ** 2, axis=2, keepdims=True))
    x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(-1, 25, 18)
    x_network = x_network.reshape(-1, 25, 18)[mask2[sex.repeat(nblocks) == 2]]
    tmp = np.abs(x_network).mean(0)
    sign = (x_network).mean(0)
    tmp[..., 5] = 0
    networkt3 = (network.iloc[np.argmax(tmp, axis=-1), 0])
    networkt3s = (sign.mean(0)[np.argmax(tmp, axis=-1)]) > 0

    dataf = pd.DataFrame({'Network-M':networkt1.values, 'Sign-M':networkt1s, 'Network-F':networkt3.values, 'Sign-F':networkt3s})
    print(dataf)

Rest Val FINA - Sex

In [ ]:
import pandas as pd
import numpy as np
from scipy.special import softmax

_, _, _, _, id2v, _, _ = visualization._get_predictions("val", "full")

# Load the network data
network = pd.read_csv("./external/schaeffer_subcortical_smith_templates.csv")
weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

# Assume visualization._get_predictions is a function that returns the required data
data, prediction, id1, id2, scores = visualization._get_predictions("pred", "full")
id1, id2, id2v = np.array(id1), np.array(id2).astype(np.int64), np.array(id2v).astype(np.int64)
keep = np.isin(id2, id2v)
data, prediction, id1, id2, scores = data.numpy()[keep], prediction.numpy()[keep], id1[keep], id2[keep], scores
scores = np.array([score.numpy() for score in scores])[keep]

# Normalize the data
x_norm = data / np.sqrt(np.sum(data ** 2, axis=1, keepdims=True))
x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(24, 480, 18)

# Initialize a DataFrame to store the results
results = pd.DataFrame(columns=['Label', 'Network-M-Init', 'Network-M-Maintain', 'Network-F-Init', 'Network-F-Maintain'])

# Define the cut-offs for T1 and T3
sex = df.sex

# Filter the population based on the cut-offs
t1_indices = np.where(sex == 1)[0]
t3_indices = np.where(sex == 2)[0]
print(t1_cutoff, t3_cutoff)
print(len(t1_indices), len(t3_indices))

# Process each label
for label in range(3):
    mask = np.argmax(softmax(prediction, axis=-1), axis=-1) == label

    # Process for T1
    vectors = []
    for i in range(len(t1_indices)):
        maskg = np.where(np.diff(mask[t1_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t1_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vectors.append(x_network[t1_indices[i]][maskg[j]:maskg[j + 1]])

    if vectors:
        vectors_init = np.concatenate([vector[:11, :] for vector in vectors], axis=0)
        vectors_init = np.abs(vectors_init).mean(0)
        vectors_init = np.argsort(vectors_init).reshape(1, 18)
        vectors_maintain = np.concatenate([vector[11:] for vector in vectors if vector.shape[0] > 10], axis=0)
        vectors_maintain = np.abs(vectors_maintain).mean(0)
        vectors_maintain = np.argsort(vectors_maintain).reshape(1, 18)
        final_init_t1 = network.iloc[:, 0].to_numpy()[vectors_init[:, -5:]].flatten()
        final_maintain_t1 = network.iloc[:, 0].to_numpy()[vectors_maintain[:, -5:]].flatten()
    else:
        final_init_t1 = final_maintain_t1 = np.array([])

    # Process for T3
    vectors = []
    for i in range(len(t3_indices)):
        maskg = np.where(np.diff(mask[t3_indices[i]]))[0] + 1
        maskg = np.concatenate([[0], maskg, [480]])
        start_idx = 0 if mask[t3_indices[i]][0] else 1
        for j in range(start_idx, len(maskg) - 1, 2):
            if maskg[j + 1] - maskg[j] > 2:
                vectors.append(x_network[t3_indices[i]][maskg[j]:maskg[j + 1]])

    if vectors:
        vectors_init = np.concatenate([vector[:11] for vector in vectors], axis=0)
        vectors_init = np.abs(vectors_init).mean(0)
        vectors_init = np.argsort(vectors_init).reshape(1, 18)
        vectors_maintain = np.concatenate([vector[11:] for vector in vectors if vector.shape[0] > 10], axis=0)
        vectors_maintain = np.abs(vectors_maintain).mean(0)
        vectors_maintain = np.argsort(vectors_maintain).reshape(1, 18)
        final_init_t3 = network.iloc[:, 0].to_numpy()[vectors_init[:, -5:]].flatten()
        final_maintain_t3 = network.iloc[:, 0].to_numpy()[vectors_maintain[:, -5:]].flatten()
    else:
        final_init_t3 = final_maintain_t3 = np.array([])

    # Append the results to the DataFrame
    results = pd.concat([results, pd.DataFrame({
        'Label': np.array(label).repeat(5),
        'Network-M-Init': final_init_t1,
        'Network-M-Maintain': final_maintain_t1,
        'Network-F-Init': final_init_t3,
        'Network-F-Maintain': final_maintain_t3
    })], ignore_index=True)

results
